# Supply Chain Control Tower – Risk Monitoring Layer

## Objective

This notebook develops the operational risk monitoring layer for the Supply Chain Control Tower.

The objective is to transform transactional demand data into actionable supply chain risk indicators that support capacity planning, bottleneck identification, inventory monitoring, and operational decision-making.

## Scope

Current Risk Modules:

- Capacity Risk

Future Risk Modules:

- Demand Risk
- Inventory Risk
- Service Risk
- Profitability Risk

## Business Questions

This notebook aims to answer the following questions:

1. Which manufacturing plants are approaching capacity constraints?

2. Where are potential bottlenecks occurring?

3. What products are driving capacity utilization increases?

4. Which operational risks require management attention?

## Data Sources

Master Data:
- dim_product
- dim_plant
- product_plant_mapping
- capacity_table

Processed Data:
- weekly_plant_demand

## Deliverables

- Capacity Utilization KPI
- Available Capacity KPI
- Capacity Alert Logic
- Bottleneck Monitoring Dataset
- Root Cause Analysis Dataset

The outputs generated in this notebook will serve as the analytical foundation for the Supply Chain Control Tower Dashboard.

### ==========================================
### Load Data
### ==========================================

In [1]:
import pandas as pd
import numpy as np

In [2]:
weekly_plant_demand = pd.read_csv(
    "../data/processed/weekly_plant_demand.csv"
)

capacity_table = pd.read_csv(
    "../data/master/capacity_table.csv"
)

df = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

product_plant_mapping = pd.read_csv(
    "../data/master/product_plant_mapping.csv"
)

In [3]:
capacity_utilization = (
    weekly_plant_demand.merge(
        capacity_table[
            ["Plant_ID", "Weekly_Capacity"]
        ],
        on="Plant_ID",
        how="left"
    )
)

capacity_utilization.head()

,Plant_ID,Order_Date,Order Item Quantity,Weekly_Capacity
0,P1,2015-01-04,567,1165
1,P1,2015-01-11,969,1165
2,P1,2015-01-18,988,1165
3,P1,2015-01-25,1009,1165
4,P1,2015-02-01,963,1165


In [4]:
# Total demand assigned to a plant in a given week

capacity_utilization = capacity_utilization.rename(
    columns={
        "Order Item Quantity": "Weekly_Demand"
    }
)

In [5]:
# Percentage of plant capacity consumed by weekly demand

capacity_utilization["Utilization"] = (
    capacity_utilization["Weekly_Demand"]
    /
    capacity_utilization["Weekly_Capacity"]
)

In [6]:
# Remaining production capacity available after fulfilling weekly demand

capacity_utilization["Available_Capacity"] = (
    capacity_utilization["Weekly_Capacity"]
    -
    capacity_utilization["Weekly_Demand"]
)

In [7]:
# Capacity risk level based on utilization threshold
def capacity_status(util):

    if util >= 0.95:
        return "Critical"

    elif util >= 0.85:
        return "Warning"

    else:
        return "Normal"

capacity_utilization["Capacity_Status"] = (
    capacity_utilization["Utilization"]
    .apply(capacity_status)
)

In [8]:
capacity_utilization.head()
capacity_utilization

,Plant_ID,Order_Date,Weekly_Demand,Weekly_Capacity,Utilization,Available_Capacity,Capacity_Status
0,P1,2015-01-04,567,1165,0.486695,598,Normal
1,P1,2015-01-11,969,1165,0.831760,196,Normal
2,P1,2015-01-18,988,1165,0.848069,177,Normal
3,P1,2015-01-25,1009,1165,0.866094,156,Warning
4,P1,2015-02-01,963,1165,0.826609,202,Normal
...,...,...,...,...,...,...,...
481,P3,2018-01-07,352,913,0.385542,561,Normal
482,P3,2018-01-14,269,913,0.294633,644,Normal
483,P3,2018-01-21,479,913,0.524644,434,Normal
484,P3,2018-01-28,304,913,0.332968,609,Normal


In [9]:
capacity_utilization

,Plant_ID,Order_Date,Weekly_Demand,Weekly_Capacity,Utilization,Available_Capacity,Capacity_Status
0,P1,2015-01-04,567,1165,0.486695,598,Normal
1,P1,2015-01-11,969,1165,0.831760,196,Normal
2,P1,2015-01-18,988,1165,0.848069,177,Normal
3,P1,2015-01-25,1009,1165,0.866094,156,Warning
4,P1,2015-02-01,963,1165,0.826609,202,Normal
...,...,...,...,...,...,...,...
481,P3,2018-01-07,352,913,0.385542,561,Normal
482,P3,2018-01-14,269,913,0.294633,644,Normal
483,P3,2018-01-21,479,913,0.524644,434,Normal
484,P3,2018-01-28,304,913,0.332968,609,Normal


In [10]:
capacity_utilization.to_csv(
    "../data/processed/capacity_utilization_weekly.csv",
    index=False
)

### ==========================================
### Product Root Cause Analysis Dataset
### ==========================================

In [11]:
plant_product_weekly_demand = (
    df.merge(
        product_plant_mapping[
            [
                "Product Card Id",
                "Plant_ID"
            ]
        ],
        on="Product Card Id",
        how="left"
    )
)

In [12]:
plant_product_weekly_demand = (
    plant_product_weekly_demand[
        [
            "order date (DateOrders)",
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order Item Quantity"
        ]
    ]
)

In [13]:
# Order week used for product-level demand monitoring

plant_product_weekly_demand[
    "Order_Date"
] = pd.to_datetime(
    plant_product_weekly_demand[
        "order date (DateOrders)"
    ]
)

In [14]:
# Weekly demand by product within each plant

plant_product_weekly_demand = (
    plant_product_weekly_demand
    .set_index("Order_Date")
    .groupby(
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name"
        ]
    )
    ["Order Item Quantity"]
    .resample("W")
    .sum()
    .reset_index()
)

In [15]:
# Total product demand assigned to a plant during a week

plant_product_weekly_demand = (
    plant_product_weekly_demand.rename(
        columns={
            "Order Item Quantity": "Weekly_Demand"
        }
    )
)

In [16]:
plant_product_weekly_demand.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand
0,P1,172,Nike Women's Tempo Shorts,2015-01-04,6
1,P1,172,Nike Women's Tempo Shorts,2015-01-11,11
2,P1,172,Nike Women's Tempo Shorts,2015-01-18,11
3,P1,172,Nike Women's Tempo Shorts,2015-01-25,18
4,P1,172,Nike Women's Tempo Shorts,2015-02-01,4


In [17]:
'''
plant_product_weekly_demand.to_csv(
    "../data/processed/plant_product_weekly_demand.csv",
    index=False
)
'''

'\nplant_product_weekly_demand.to_csv(\n    "../data/processed/plant_product_weekly_demand.csv",\n    index=False\n)\n'

### ==========================================
### Top Demand Product Driver
### ==========================================

In [36]:
plant_product_weekly_demand = pd.read_csv(r'../data/processed/plant_product_weekly_demand.csv')

In [37]:
# Product demand ranking within each plant and week

top_product_driver = (
    plant_product_weekly_demand
    .sort_values(
        [
            "Plant_ID",
            "Order_Date",
            "Weekly_Demand"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

In [38]:
# Demand contribution rank within a plant-week

top_product_driver["Demand_Rank"] = (
    top_product_driver
    .groupby(
        [
            "Plant_ID",
            "Order_Date"
        ]
    )["Weekly_Demand"]
    .rank(
        method="dense",
        ascending=False
    )
)

### ==========================================
### Weekly Demand Change
### ==========================================

In [39]:
# Ensure product demand history is ordered chronologically

plant_product_weekly_demand = (
    plant_product_weekly_demand
    .sort_values(
        [
            "Plant_ID",
            "Product Card Id",
            "Order_Date"
        ]
    )
)

In [40]:
# Product demand observed in the previous week

plant_product_weekly_demand["Demand_Last_Week"] = (
    plant_product_weekly_demand
    .groupby(
        [
            "Plant_ID",
            "Product Card Id"
        ]
    )["Weekly_Demand"]
    .shift(1)
)

In [41]:
# Absolute increase in demand compared with previous week

plant_product_weekly_demand["Demand_Change"] = (
    plant_product_weekly_demand["Weekly_Demand"]
    -
    plant_product_weekly_demand["Demand_Last_Week"]
)

In [42]:
# Week-over-week demand growth percentage

plant_product_weekly_demand["Demand_Growth"] = (
    plant_product_weekly_demand["Demand_Change"]
    /
    plant_product_weekly_demand["Demand_Last_Week"]
)

In [43]:
plant_product_weekly_demand = (
    plant_product_weekly_demand
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [44]:
plant_product_weekly_demand[
    [
        "Plant_ID",
        "Product Name",
        "Order_Date",
        "Weekly_Demand",
        "Demand_Last_Week",
        "Demand_Change",
        "Demand_Growth"
    ]
].head(20)

,Plant_ID,Product Name,Order_Date,Weekly_Demand,Demand_Last_Week,Demand_Change,Demand_Growth
0,P1,Nike Women's Tempo Shorts,2015-01-04,6,NaN,NaN,NaN
1,P1,Nike Women's Tempo Shorts,2015-01-11,11,6.0,5.0,0.833333
2,P1,Nike Women's Tempo Shorts,2015-01-18,11,11.0,0.0,0.000000
3,P1,Nike Women's Tempo Shorts,2015-01-25,18,11.0,7.0,0.636364
4,P1,Nike Women's Tempo Shorts,2015-02-01,4,18.0,-14.0,-0.777778
5,P1,Nike Women's Tempo Shorts,2015-02-08,9,4.0,5.0,1.250000
6,P1,Nike Women's Tempo Shorts,2015-02-15,9,9.0,0.0,0.000000
7,P1,Nike Women's Tempo Shorts,2015-02-22,9,9.0,0.0,0.000000
8,P1,Nike Women's Tempo Shorts,2015-03-01,5,9.0,-4.0,-0.444444
9,P1,Nike Women's Tempo Shorts,2015-03-08,6,5.0,1.0,0.200000


In [45]:
top_product_driver = (
    top_product_driver.merge(
        plant_product_weekly_demand[
            [
                "Plant_ID",
                "Product Card Id",
                "Order_Date",
                "Demand_Last_Week",
                "Demand_Change",
                "Demand_Growth"
            ]
        ],
        on=[
            "Plant_ID",
            "Product Card Id",
            "Order_Date"
        ],
        how="left"
    )
)

In [46]:
# Top 10 demand-contributing products within a plant-week

top_product_driver = (
    top_product_driver[
        top_product_driver["Demand_Rank"] <= 10
    ]
)

In [47]:
top_product_driver.head(20)


,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Demand_Rank,Demand_Last_Week,Demand_Change,Demand_Growth
0,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-04,306,1.0,NaN,NaN,NaN
1,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-04,160,2.0,NaN,NaN,NaN
2,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-04,77,3.0,NaN,NaN,NaN
3,P1,276,Under Armour Women's Ignite Slide,2015-01-04,10,4.0,NaN,NaN,NaN
4,P1,172,Nike Women's Tempo Shorts,2015-01-04,6,5.0,NaN,NaN,NaN
5,P1,235,Under Armour Hustle Storm Medium Duffle Bag,2015-01-04,4,6.0,NaN,NaN,NaN
6,P1,278,Under Armour Men's Compression EV SL Slide,2015-01-04,4,6.0,NaN,NaN,NaN
7,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-11,529,1.0,306.0,223.0,0.728758
8,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-11,243,2.0,160.0,83.0,0.518750
9,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-11,147,3.0,77.0,70.0,0.909091


In [30]:
top_product_driver.groupby(
    ["Plant_ID", "Order_Date"]
).size().describe()

count    481.000000
mean       9.852391
std        5.718486
min        1.000000
25%        5.000000
50%       10.000000
75%       13.000000
max       31.000000
dtype: float64

### ==========================================
### Demand Growth Driver
### ==========================================

In [50]:
# Total plant demand during a given week

plant_weekly_total_demand = (
    plant_product_weekly_demand
    .groupby(
        [
            "Plant_ID",
            "Order_Date"
        ]
    )["Weekly_Demand"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Weekly_Demand": "Plant_Weekly_Demand"
        }
    )
)

In [51]:
top_product_driver = (
    top_product_driver.merge(
        plant_weekly_total_demand,
        on=[
            "Plant_ID",
            "Order_Date"
        ],
        how="left"
    )
)

In [52]:
# Percentage contribution to total plant demand

top_product_driver["Demand_Contribution"] = (
    top_product_driver["Weekly_Demand"]
    /
    top_product_driver["Plant_Weekly_Demand"]
)

In [53]:
top_product_driver.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Demand_Rank,Demand_Last_Week,Demand_Change,Demand_Growth,Plant_Weekly_Demand,Demand_Contribution
0,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-04,306,1.0,NaN,NaN,NaN,567,0.539683
1,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-04,160,2.0,NaN,NaN,NaN,567,0.282187
2,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-04,77,3.0,NaN,NaN,NaN,567,0.135802
3,P1,276,Under Armour Women's Ignite Slide,2015-01-04,10,4.0,NaN,NaN,NaN,567,0.017637
4,P1,172,Nike Women's Tempo Shorts,2015-01-04,6,5.0,NaN,NaN,NaN,567,0.010582


In [54]:
# Percentage contribution to total plant demand

top_product_driver["Demand_Contribution_Pct"] = (
    top_product_driver["Demand_Contribution"]
    * 100
).round(2)

In [ ]:
# Cumulative demand contribution within a plant-week
top_product_driver = (
    top_product_driver
    .sort_values(
        [
            "Plant_ID",
            "Order_Date",
            "Demand_Rank"
        ]
    )
)



top_product_driver["Cumulative_Contribution"] = (
    top_product_driver
    .groupby(
        [
            "Plant_ID",
            "Order_Date"
        ]
    )["Demand_Contribution"]
    .cumsum()
)

In [56]:
top_product_driver.columns

Index(['Plant_ID', 'Product Card Id', 'Product Name', 'Order_Date',
       'Weekly_Demand', 'Demand_Rank', 'Demand_Last_Week', 'Demand_Change',
       'Demand_Growth', 'Plant_Weekly_Demand', 'Demand_Contribution',
       'Demand_Contribution_Pct', 'Cumulative_Contribution'],
      dtype='object')

In [57]:
top_product_driver.head(20)

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Demand_Rank,Demand_Last_Week,Demand_Change,Demand_Growth,Plant_Weekly_Demand,Demand_Contribution,Demand_Contribution_Pct,Cumulative_Contribution
0,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-04,306,1.0,NaN,NaN,NaN,567,0.539683,53.97,0.539683
1,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-04,160,2.0,NaN,NaN,NaN,567,0.282187,28.22,0.821869
2,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-04,77,3.0,NaN,NaN,NaN,567,0.135802,13.58,0.957672
3,P1,276,Under Armour Women's Ignite Slide,2015-01-04,10,4.0,NaN,NaN,NaN,567,0.017637,1.76,0.975309
4,P1,172,Nike Women's Tempo Shorts,2015-01-04,6,5.0,NaN,NaN,NaN,567,0.010582,1.06,0.985891
5,P1,235,Under Armour Hustle Storm Medium Duffle Bag,2015-01-04,4,6.0,NaN,NaN,NaN,567,0.007055,0.71,0.992945
6,P1,278,Under Armour Men's Compression EV SL Slide,2015-01-04,4,6.0,NaN,NaN,NaN,567,0.007055,0.71,1.000000
7,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-11,529,1.0,306.0,223.0,0.728758,969,0.545924,54.59,0.545924
8,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-11,243,2.0,160.0,83.0,0.518750,969,0.250774,25.08,0.796698
9,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-11,147,3.0,77.0,70.0,0.909091,969,0.151703,15.17,0.948400


In [ ]:
"""
top_product_driver.to_csv(
    "../data/processed/top_product_driver_weekly.csv",
    index=False
)
"""